In [1]:
import importlib
import sys

# The exact list of libraries your project logic uses
required_libs = [
    # Core Deep Learning
    ("torch", "PyTorch"),
    ("torchvision", "TorchVision"),
    ("timm", "Timm (EfficientNet backbone)"),
    
    # Texture & Image Processing
    ("cv2", "OpenCV (FFT/LBP maps)"),
    ("numpy", "NumPy"),
    ("PIL", "Pillow (Image loading)"),
    
    # Metrics & Evaluation
    ("sklearn", "Scikit-Learn (AUC/EER)"),
    ("matplotlib", "Matplotlib (ROC Curves)"),
    
    # Utilities
    ("datasets", "HuggingFace Datasets"),
    ("pandas", "Pandas")
]

print(f"{'Library':<20} | {'Status':<15}")
print("-" * 40)

for import_name, description in required_libs:
    try:
        importlib.import_module(import_name)
        print(f"{import_name:<20} | ✅ Ready")
    except ImportError:
        print(f"{import_name:<20} | ❌ MISSING")

print("\nIf 'datasets' is the only one missing, your code might still run")
print("if you are loading images directly from your student folder.")

Library              | Status         
----------------------------------------
torch                | ✅ Ready
torchvision          | ✅ Ready


/home/teaching/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


timm                 | ✅ Ready
cv2                  | ✅ Ready
numpy                | ✅ Ready
PIL                  | ✅ Ready
sklearn              | ✅ Ready
matplotlib           | ✅ Ready
datasets             | ✅ Ready
pandas               | ✅ Ready

If 'datasets' is the only one missing, your code might still run
if you are loading images directly from your student folder.


In [2]:
import shutil
import os

# The folder we want to keep clean
local_pkg_path = os.path.join(os.getcwd(), "my_packages")

# List of problematic sub-folders to wipe out
problem_folders = ['fsspec', 'numpy', 'typing_extensions']

for folder in problem_folders:
    target = os.path.join(local_pkg_path, folder)
    if os.path.exists(target):
        shutil.rmtree(target)
        print(f"Cleared: {target}")

print("✅ Local conflicts removed. Now you MUST restart the kernel.")

Cleared: /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/my_packages/fsspec
Cleared: /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/my_packages/numpy
✅ Local conflicts removed. Now you MUST restart the kernel.


In [3]:
import torch
torch.set_num_threads(4)   # or 8 if CPU is strong

In [4]:
import os, math, random, shutil
import numpy as np
from pathlib import Path
from PIL import Image
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import cv2
 
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for remote server
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
 
# ── Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
 
# ══════════════════════════════════════════════════════════════════
# DATASET PATHS  ← adjust BASE if your working directory differs
# ══════════════════════════════════════════════════════════════════
BASE       = 'DL_ProjectP12_clean'
REAL_DIR   = os.path.join(BASE, 'real')
ATTACK_DIR = os.path.join(BASE, 'attack')   # attack/ subfolders
SAVE_DIR   = os.path.join(BASE, 'outputs')  # results saved here
os.makedirs(SAVE_DIR, exist_ok=True)
 
# ── Hyperparameters
IMG_SIZE   = 224
PATCH_SIZE = 112        # 4 non-overlapping 112×112 patches per image
BATCH_SIZE = 4
PHASE1_EP  = 100
PHASE2_EP  = 100
LR1        = 1e-3
LR2        = 5e-5
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device  : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print(f'Base    : {os.path.abspath(BASE)}')
print(f'Real    : {os.path.abspath(REAL_DIR)}')
print(f'Attack  : {os.path.abspath(ATTACK_DIR)}')
print(f'Outputs : {os.path.abspath(SAVE_DIR)}')

# ✅ Add this
torch.set_num_threads(4)

Device  : cuda
GPU     : NVIDIA RTX A5000
VRAM    : 25.4 GB
Base    : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean
Real    : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/real
Attack  : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/attack
Outputs : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/outputs


In [5]:
# ── Cell 3: Verify Dataset Paths ─────────────────────────────────
# Checks that the folder structure matches what is expected.
# If this cell fails → adjust BASE above.
 
assert os.path.isdir(REAL_DIR),   f'Not found: {REAL_DIR}'
assert os.path.isdir(ATTACK_DIR), f'Not found: {ATTACK_DIR}'
 
def count_subjects_and_images(folder):
    subjects = sorted([
        s for s in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, s)) and 'FH' not in s
    ])

    total_imgs = 0

    for s in subjects:
        subj_path = os.path.join(folder, s)

        for root, dirs, files in os.walk(subj_path):
            total_imgs += sum(
                1 for f in files
                if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))
            )

    return subjects, total_imgs
 
real_subjects,   real_imgs   = count_subjects_and_images(REAL_DIR)
attack_subjects, attack_imgs = count_subjects_and_images(ATTACK_DIR)
 
print(f'\nDataset verified:')
print(f'  real/   → {len(real_subjects):4d} subjects,  {real_imgs:6d} images')
print(f'            first: {real_subjects[:3]}  last: {real_subjects[-2:]}')
print(f'  attack/ → {len(attack_subjects):4d} subjects,  {attack_imgs:6d} images')
print(f'            first: {attack_subjects[:3]}  last: {attack_subjects[-2:]}')
print(f'\nTotal images: {real_imgs + attack_imgs}')


Dataset verified:
  real/   →  295 subjects,    5900 images
            first: ['248', '249', '250']  last: ['541', '542']
  attack/ →  295 subjects,    5900 images
            first: ['248', '249', '250']  last: ['541', '542']

Total images: 11800


In [6]:
import os
print(os.getcwd())

/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY


In [7]:
print(os.listdir())

['train_pad_vscode_cells_fixed.ipynb', '.codex', 'resnet18.ipynb', 'class_subject.ipynb', 'best_attn_pad.pt', 'train_pad_vscode_cells.ipynb', 'attention_pad_results.png', 'my_packages', 'DL_ProjectP12_clean', 'DL-12']


In [8]:
import os

print("Inside BASE:")
print(os.listdir('DL_ProjectP12_clean'))

Inside BASE:
['attack', 'outputs_295class', 'real', 'outputs']


In [9]:
# -- Cell 4: Texture Feature Helpers (FIXED: fast LBP via skimage) --
from skimage.feature import local_binary_pattern as skimage_lbp

def compute_lbp(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    # skimage LBP: identical 8-neighbour radius-1 pattern, ~100x faster than pure Python loop
    lbp  = skimage_lbp(gray, P=8, R=1, method='default').astype('uint8')
    hist, _ = np.histogram(lbp.ravel(), bins=256, range=(0, 256))
    hist     = hist.astype('float32')
    hist    /= hist.sum() + 1e-7
    return hist   # (256,)

def compute_hf_map(img_np, sigma=2.0):
    img_f = img_np.astype('float32') / 255.0
    blur  = cv2.GaussianBlur(img_f, (0, 0), sigma)
    hf    = img_f - blur
    hf    = (hf - hf.min()) / (hf.max() - hf.min() + 1e-7)
    return torch.tensor(np.transpose(hf, (2, 0, 1)), dtype=torch.float32)

def compute_fft_map(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY).astype('float32')
    mag  = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray))))
    mag  = (mag - mag.min()) / (mag.max() - mag.min() + 1e-7)
    return torch.tensor(mag[np.newaxis], dtype=torch.float32)

print('Texture helpers ready (LBP via skimage [fast], HF map, FFT map).')


Texture helpers ready (LBP via skimage [fast], HF map, FFT map).


In [10]:
# # ── Cell 5: Dataset with Patch Extraction (Fixed for Nested Folders) ──────────
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# from PIL import Image
# from pathlib import Path
# from collections import defaultdict
# from sklearn.model_selection import train_test_split

# REQUIRED_COUNT = 20 

# def extract_patches(img_tensor, patch_size=PATCH_SIZE):
#     patches = []
#     for i in range(0, IMG_SIZE, patch_size):
#         for j in range(0, IMG_SIZE, patch_size):
#             patches.append(img_tensor[:, i:i+patch_size, j:j+patch_size])
#     return torch.stack(patches, dim=0)

# class TexturePatchDataset(Dataset):
#     def __init__(self, samples, rgb_transform):
#         self.samples = samples
#         self.rgb_tf  = rgb_transform
#         self.resize  = transforms.Resize((IMG_SIZE, IMG_SIZE))
#     def __len__(self):
#         return len(self.samples)
#     def __getitem__(self, idx):
#         path, label = self.samples[idx]
#         pil  = self.resize(Image.open(path).convert('RGB'))
#         np_  = np.array(pil)
#         rgb     = self.rgb_tf(pil)
#         patches = extract_patches(rgb)
#         hf      = compute_hf_map(np_)
#         fft     = compute_fft_map(np_)
#         lbp     = torch.tensor(compute_lbp(np_), dtype=torch.float32)
#         return (rgb, patches, hf, fft, lbp, torch.tensor(label, dtype=torch.float32))

# def build_filtered_samples(real_dir, attack_dir, req_count=20):
#     subject_map = defaultdict(lambda: {'real': [], 'attack': []})
#     exts = ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP')

#     def scan_for_subjects(base_path, key):
#         base = Path(base_path)
#         # This searches for folders that contain images, specifically looking for the numbered folders
#         for img_path in base.rglob('*'):
#             if img_path.suffix.lower() in exts:
#                 # The parent folder name (e.g., '246') is our subject ID
#                 subj_id = img_path.parent.name
#                 # Exclude unwanted folder names
#                 if 'FH' not in subj_id and 'dataset' not in subj_id:
#                     subject_map[subj_id][key].append(str(img_path))

#     print("Scanning directories... (this may take a moment)")
#     scan_for_subjects(real_dir, 'real')
#     scan_for_subjects(attack_dir, 'attack')

#     valid_samples = []
#     valid_subjects = []

#     print(f"\n--- Filtering Results (Target: {req_count} images) ---")
#     # Only iterate over subjects found in BOTH
#     all_possible_subjects = sorted(subject_map.keys())
    
#     for subj in all_possible_subjects:
#         r_imgs = subject_map[subj]['real']
#         a_imgs = subject_map[subj]['attack']
        
#         # Check if the counts match your 20-image requirement
#         if len(r_imgs) == req_count and len(a_imgs) == req_count:
#             valid_subjects.append(subj)
#             for r in r_imgs: valid_samples.append((r, 0, subj))
#             for a in a_imgs: valid_samples.append((a, 1, subj))
        
#     return valid_samples, valid_subjects

# # 1. Build list
# samples, valid_subjects = build_filtered_samples(REAL_DIR, ATTACK_DIR, REQUIRED_COUNT)

# # 2. Safety Check
# if not valid_subjects:
#     print("❌ Error: Still no subjects found.")
#     print("Check: Are your folders named like '246' and do they contain exactly 20 images?")
#     # Let's print one example of what was found to help you debug
#     example_subj = list(samples)[:1]
#     raise ValueError("Zero subjects matched the criteria.")

# # 3. Subject-Wise Split
# train_s, temp_s = train_test_split(valid_subjects, test_size=0.30, random_state=SEED)
# val_s, test_s   = train_test_split(temp_s, test_size=0.50, random_state=SEED)

# train_data = [(p, l) for (p, l, s) in samples if s in train_s]
# val_data   = [(p, l) for (p, l, s) in samples if s in val_s]
# test_data  = [(p, l) for (p, l, s) in samples if s in test_s]

# print(f'Done! Found {len(valid_subjects)} matching subjects.')
# print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

# # ... [Keep your DataLoader and Transform code from the previous response] ...

# ── Cell 5: Dataset with Patch Extraction (Subject-Wise & Fixed for Nested Folders) ──
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

# Enforce your rule: Subject must have exactly 20 real and 20 attack images
REQUIRED_COUNT = 20 

def extract_patches(img_tensor, patch_size=PATCH_SIZE):
    """Split (3, 224, 224) tensor into 4 non-overlapping patches."""
    patches = []
    for i in range(0, IMG_SIZE, patch_size):
        for j in range(0, IMG_SIZE, patch_size):
            patches.append(img_tensor[:, i:i+patch_size, j:j+patch_size])
    return torch.stack(patches, dim=0)

class TexturePatchDataset(Dataset):
    def __init__(self, samples, rgb_transform):
        self.samples = samples
        self.rgb_tf  = rgb_transform
        self.resize  = transforms.Resize((IMG_SIZE, IMG_SIZE))
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        pil  = self.resize(Image.open(path).convert('RGB'))
        np_  = np.array(pil)
        
        rgb     = self.rgb_tf(pil)
        patches = extract_patches(rgb)
        hf      = compute_hf_map(np_)
        fft     = compute_fft_map(np_)
        lbp     = torch.tensor(compute_lbp(np_), dtype=torch.float32)
        
        return (rgb, patches, hf, fft, lbp, 
                torch.tensor(label, dtype=torch.float32))

def build_filtered_samples(real_dir, attack_dir, req_count=20):
    subject_map = defaultdict(lambda: {'real': [], 'attack': []})
    exts = ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP')

    def scan_for_subjects(base_path, key):
        base = Path(base_path)
        # Recursive scan to find nested folders like '246', '247'
        for img_path in base.rglob('*'):
            if img_path.suffix.lower() in exts:
                subj_id = img_path.parent.name
                if 'FH' not in subj_id and 'dataset' not in subj_id:
                    subject_map[subj_id][key].append(str(img_path))

    print("Scanning directories... (this may take a moment)")
    scan_for_subjects(real_dir, 'real')
    scan_for_subjects(attack_dir, 'attack')

    valid_samples = []
    valid_subjects = []

    for subj in sorted(subject_map.keys()):
        r_imgs = subject_map[subj]['real']
        a_imgs = subject_map[subj]['attack']
        
        # Enforce exact matching (Subject 246/248 rule)
        if len(r_imgs) == req_count and len(a_imgs) == req_count:
            valid_subjects.append(subj)
            for r in r_imgs: valid_samples.append((r, 0, subj))
            for a in a_imgs: valid_samples.append((a, 1, subj))
        
    return valid_samples, valid_subjects

# 1. Build the sample list
samples, valid_subjects = build_filtered_samples(REAL_DIR, ATTACK_DIR, REQUIRED_COUNT)

# 2. Safety Check
if not valid_subjects:
    raise ValueError(f"No subjects found with exactly {REQUIRED_COUNT} images in both folders.")

# 3. SUBJECT-WISE SPLIT (Dividing by people, not images)
train_s, temp_s = train_test_split(valid_subjects, test_size=0.30, random_state=SEED)
val_s, test_s   = train_test_split(temp_s, test_size=0.50, random_state=SEED)

train_data = [(p, l) for (p, l, s) in samples if s in train_s]
val_data   = [(p, l) for (p, l, s) in samples if s in val_s]
test_data  = [(p, l) for (p, l, s) in samples if s in test_s]

print(f'Done! Found {len(valid_subjects)} matching subjects.')
print(f'Train: {len(train_data)} images | Val: {len(val_data)} | Test: {len(test_data)}')

# ── Transforms & DataLoaders ──────────────────────────
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Optimized for your NVIDIA RTX A5000
NUM_WORKERS = 4  

train_loader = DataLoader(
    TexturePatchDataset(train_data, train_tf), 
    batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    TexturePatchDataset(val_data, eval_tf), 
    batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    TexturePatchDataset(test_data, eval_tf), 
    batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'\nDataLoaders ready using {NUM_WORKERS} workers.')
print(type(train_loader))

Scanning directories... (this may take a moment)
Done! Found 295 matching subjects.
Train: 8240 images | Val: 1760 | Test: 1800

DataLoaders ready using 4 workers.
<class 'torch.utils.data.dataloader.DataLoader'>


In [11]:
# ── Cell 6: SE Attention Block ────────────────────────────────────
# ─────────────────────────────────────────────────────────────────
# SQUEEZE-AND-EXCITATION (SE) BLOCK:
#
# What it does:
#   Standard CNN treats all feature channels equally.
#   SE block learns to WEIGHT each channel differently.
#   Channels that detect print artifacts → HIGH weight
#   Channels that detect irrelevant texture → LOW weight
#
# How it works:
#   1. Squeeze: global average pool → compress (C,H,W) to (C,)
#      "What is the average activation of each channel?"
#   2. Excitation: small FC network → output weight per channel
#      FC: C → C//16 → C → Sigmoid → weights ∈ (0,1)
#      This is a lightweight attention mechanism (~2K params)
#   3. Scale: multiply original feature map by channel weights
#      Channels detecting print noise get amplified.
#      Channels detecting irrelevant texture get suppressed.
#
# Why reduction=16?
#   C//16 bottleneck keeps params small (e.g. 64 → 4 → 64)
#   while still learning meaningful channel relationships.
#
# Example:
#   Channel 23 detects moiré patterns → SE gives it weight 0.95
#   Channel 47 detects generic edges  → SE gives it weight 0.12
#   Result: model now "focuses" on moiré regions
# ─────────────────────────────────────────────────────────────────
 
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block.
    Learns per-channel attention weights → focuses on artifact channels.
 
    Args:
        channels  : number of input channels
        reduction : bottleneck ratio (default 16)
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        # Ensure bottleneck is at least 1
        mid = max(channels // reduction, 1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),    # squeeze: C → C//16
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),    # excite:  C//16 → C
            nn.Sigmoid()                 # weights in (0,1)
        )
 
    def forward(self, x):
        # x: (B, C, H, W)
        b, c, _, _ = x.size()
        # Squeeze: global average pool → (B, C)
        y = x.mean(dim=(2, 3))
        # Excite: learn channel importance weights → (B, C, 1, 1)
        y = self.fc(y).view(b, c, 1, 1)
        # Scale: reweight feature maps
        return x * y
 

In [12]:
# ── Cell 7: Full Architecture ─────────────────────────────────────
# ─────────────────────────────────────────────────────────────────
# FOUR-BRANCH ARCHITECTURE:
#
# Branch 1 — RGB (EfficientNet-B0):
#   Full 224×224 image → global identity + coarse appearance
#   Output: 256-d
#
# Branch 2 — Texture (TextureCNN with SE attention):
#   HF + FFT (4 channels) → fine print artifact detection
#   SE blocks focus channels on artifact-specific patterns
#   Output: 128-d
#
# Branch 3 — LBP (MLP):
#   256-d LBP histogram → statistical micro-texture
#   Output: 64-d
#
# Branch 4 — Patch (lightweight CNN):  ← NEW
#   4 patches of 112×112 → local artifact detection
#   Each patch processed independently, then mean+max pooled
#   Output: 64-d
#
# Fusion: [256 + 128 + 64 + 64] = 512-d
#   → L2-normalize (cosine-ready embedding)
#   → classifier → logit
# ─────────────────────────────────────────────────────────────────
 
class TextureCNNWithSE(nn.Module):
    """
    Texture CNN with SE attention at each block.
    Input: 4 channels (3 HF + 1 FFT), size 224×224.
    SE blocks amplify artifact-detecting channels,
    suppress irrelevant texture channels.
    Output: 128-d embedding.
    """
    def __init__(self):
        super().__init__()
 
        # Block 1: fine edges + noise (3×3 kernel for fine texture)
        self.block1 = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.se1 = SEBlock(32, reduction=8)   # attention after block 1
        self.pool1 = nn.MaxPool2d(2)          # 224 → 112
 
        # Block 2: medium patterns (moiré, halftone)
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.se2 = SEBlock(64, reduction=8)   # attention after block 2
        self.pool2 = nn.MaxPool2d(2)          # 112 → 56
 
        # Block 3: coarser print structure
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.se3 = SEBlock(128, reduction=16)  # attention after block 3
        self.gap  = nn.AdaptiveAvgPool2d((4, 4))
 
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
        )
 
    def forward(self, x):
        x = self.pool1(self.se1(self.block1(x)))   # SE after block 1
        x = self.pool2(self.se2(self.block2(x)))   # SE after block 2
        x = self.gap(self.se3(self.block3(x)))     # SE after block 3
        return self.projector(x)
 
 
class LBPBranch(nn.Module):
    """MLP on 256-d LBP histogram → 64-d."""
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 64),
        )
    def forward(self, x): return self.mlp(x)
 
 
class PatchBranch(nn.Module):
    """
    Lightweight CNN for patch-level artifact detection.
    Processes each of the 4 patches (112×112) independently.
    Uses mean pooling + max pooling across patches then concatenates.
    Mean pool: average artifact score across patches.
    Max pool:  catch the single most suspicious patch.
    Output: 64-d (32 mean + 32 max).
 
    Why mean + max?
      Mean: tells you the average level of spoofing across patches.
      Max:  tells you the worst (most suspicious) single patch.
      Together: more informative than either alone.
    """
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16),
            nn.ReLU(inplace=True), nn.MaxPool2d(2),   # 112 → 56
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2),   # 56 → 28
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3)),              # → 64×3×3
            nn.Flatten(),                              # → 576
            nn.Linear(576, 32),
        )
 
    def forward(self, patches):
        # patches: (B, 4, 3, 112, 112)
        B, N, C, H, W = patches.shape
        # Reshape to process all patches at once: (B*4, 3, 112, 112)
        x = patches.view(B * N, C, H, W)
        x = self.cnn(x)                   # (B*4, 32)
        x = x.view(B, N, 32)              # (B, 4, 32)
 
        # Mean pool: average spoof signal across 4 patches
        mean_feat = x.mean(dim=1)         # (B, 32)
        # Max pool: most suspicious single patch
        max_feat  = x.max(dim=1).values   # (B, 32)
 
        return torch.cat([mean_feat, max_feat], dim=1)  # (B, 64)
 
 
class AttentionPAD(nn.Module):
    """
    Four-branch texture-aware PAD with SE attention + patch learning.
 
    Branches:
      1. RGB → EfficientNet-B0 → 256-d
      2. HF+FFT → TextureCNN+SE → 128-d
      3. LBP histogram → MLP → 64-d
      4. 4 patches → PatchBranch → 64-d
 
    Fusion: concat(256+128+64+64)=512-d → L2-normalize → classifier
    """
    def __init__(self):
        super().__init__()
 
        # Branch 1: RGB backbone
        self.rgb_backbone = timm.create_model(
            'efficientnet_b0', pretrained=True, num_classes=0
        )
        self.rgb_proj = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.35),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
        )
 
        # Branch 2: Texture with SE attention
        self.texture_branch = TextureCNNWithSE()   # → 128-d
 
        # Branch 3: LBP
        self.lbp_branch = LBPBranch()              # → 64-d
 
        # Branch 4: Patch
        self.patch_branch = PatchBranch()          # → 64-d
 
        # Fusion: 256 + 128 + 64 + 64 = 512-d
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
 
    def forward(self, rgb, patches, hf, fft, lbp):
        # Branch 1
        rgb_emb = self.rgb_proj(self.rgb_backbone(rgb))         # (B,256)
        # Branch 2
        tex_emb = self.texture_branch(torch.cat([hf, fft], 1)) # (B,128)
        # Branch 3
        lbp_emb = self.lbp_branch(lbp)                         # (B,64)
        # Branch 4
        pat_emb = self.patch_branch(patches)                    # (B,64)
 
        # Fusion + L2 normalize (cosine-ready embedding)
        fused     = torch.cat([rgb_emb, tex_emb, lbp_emb, pat_emb], dim=1)  # (B,512)
        embedding = F.normalize(fused, p=2, dim=1)  # onto unit hypersphere
        logit     = self.classifier(embedding).squeeze(1)
        return logit, embedding
 
    def freeze_backbone(self):
        for p in self.rgb_backbone.parameters():
            p.requires_grad = False
 
    def unfreeze_last_blocks(self, n_blocks=2):
        for name, p in self.rgb_backbone.named_parameters():
            if 'blocks' in name:
                idx = int(name.split('blocks.')[1].split('.')[0])
                p.requires_grad = (idx >= 7 - n_blocks)
            elif any(x in name for x in ['conv_head', 'bn2']):
                p.requires_grad = True
            else:
                p.requires_grad = False
 
 
model = AttentionPAD().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'AttentionPAD loaded — {total/1e6:.2f}M params')
print('  Branch 1: RGB → EfficientNet-B0 → 256-d')
print('  Branch 2: HF+FFT → TextureCNN+SE(attention) → 128-d')
print('  Branch 3: LBP → MLP → 64-d')
print('  Branch 4: 4 patches → PatchBranch(mean+max) → 64-d')
print('  Fusion  : 512-d → L2-normalize → cosine space')

AttentionPAD loaded — 5.75M params
  Branch 1: RGB → EfficientNet-B0 → 256-d
  Branch 2: HF+FFT → TextureCNN+SE(attention) → 128-d
  Branch 3: LBP → MLP → 64-d
  Branch 4: 4 patches → PatchBranch(mean+max) → 64-d
  Fusion  : 512-d → L2-normalize → cosine space


In [13]:
# ── Cell 8: Combined Loss (Focal + ArcFace + Contrastive) ────────
# ─────────────────────────────────────────────────────────────────
# CONTRASTIVE CONSTRAINT:
#
# ArcFace pushes classes apart on a hypersphere via angular margin.
# Contrastive loss adds an EXPLICIT pair-wise constraint:
#
#   Real–Real pair:  cosine similarity should be HIGH → loss if too low
#   Real–Spoof pair: cosine similarity should be LOW  → loss if too high
#
# Formula:
#   L_contrast = (1/N²) × Σ_pairs [
#     y × max(0, margin - cos(a,b))²          ← same class: push close
#     + (1-y) × max(0, cos(a,b) - margin)²   ← diff class: push apart
#   ]
#
# Why this helps:
#   ArcFace works on classification (each sample vs class center).
#   Contrastive works on PAIRS (sample vs sample).
#   Together they enforce both global and local separation.
#
# margin=0.5 means:
#   Same class pairs should have cosine > 0.5
#   Different class pairs should have cosine < 0.5
# ─────────────────────────────────────────────────────────────────
 
class CombinedPADLoss(nn.Module):
    """
    Focal + ArcFace + Contrastive.
 
    focal_w      : weight for focal loss term      (default 0.5)
    contrastive_w: weight for contrastive term      (default 0.3)
    arc margin   : m=0.50 (angular margin ≈ 28.6°)
    """
    def __init__(self, embedding_dim=512, num_classes=2,
                 s=30.0, m=0.50, gamma=2.0, alpha=0.25,
                 contrastive_margin=0.5,
                 focal_w=0.5, contrastive_w=0.3):
        super().__init__()
        # ArcFace
        self.s = s; self.m = m
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        # Focal
        self.gamma = gamma; self.alpha = alpha
        self.bce   = nn.BCEWithLogitsLoss(reduction='none')
        # Contrastive
        self.cont_margin = contrastive_margin
        # Weights
        self.focal_w = focal_w
        self.cont_w  = contrastive_w
 
    def focal_loss(self, logits, labels):
        bce_raw = self.bce(logits, labels)
        probs   = torch.sigmoid(logits)
        pt      = torch.where(labels == 1, probs, 1 - probs)
        alpha_t = torch.where(labels == 1,
                               torch.full_like(labels, self.alpha),
                               torch.full_like(labels, 1 - self.alpha))
        return (alpha_t * (1 - pt) ** self.gamma * bce_raw).mean()
 
    def arcface_loss(self, embeddings, labels):
        W      = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, W)
        sine   = torch.sqrt(torch.clamp(1.0 - cosine**2, min=1e-6))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        int_l  = labels.long()
        oh     = torch.zeros_like(cosine)
        oh.scatter_(1, int_l.view(-1, 1), 1.0)
        out    = (oh * phi) + ((1 - oh) * cosine)
        return F.cross_entropy(out * self.s, int_l)
 
    def contrastive_loss(self, embeddings, labels):
        """
        Pair-wise contrastive loss on a mini-batch.
        Same-class pairs (y=1): penalize if cosine < margin.
        Diff-class pairs (y=0): penalize if cosine > margin.
        Uses all pairs within the batch.
        """
        # Pairwise cosine similarity matrix: (B, B)
        # embeddings are already L2-normalized
        sim   = torch.mm(embeddings, embeddings.t())   # (B,B)
        # Pairwise label match matrix: 1 if same class, 0 if different
        lmat  = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()  # (B,B)
 
        # Same-class loss: penalize when cosine < margin
        same_loss = lmat       * F.relu(self.cont_margin - sim) ** 2
        # Diff-class loss: penalize when cosine > margin
        diff_loss = (1 - lmat) * F.relu(sim - self.cont_margin) ** 2
 
        # Exclude diagonal (self-similarity = 1.0, always same class)
        mask = ~torch.eye(len(labels), dtype=torch.bool, device=labels.device)
        return (same_loss[mask] + diff_loss[mask]).mean()
 
    def forward(self, logits, labels, embeddings=None):
        fl = self.focal_loss(logits, labels)
        if embeddings is None:
            return fl
        al = self.arcface_loss(embeddings, labels)
        cl = self.contrastive_loss(embeddings, labels)
        # Total = ArcFace + focal_w*Focal + cont_w*Contrastive
        return al + self.focal_w * fl + self.cont_w * cl
 
criterion = CombinedPADLoss(embedding_dim=512).to(DEVICE)
print('CombinedPADLoss ready:')
print('  ArcFace      : m=0.50, s=30 (angular margin)')
print('  Focal Loss   : gamma=2, alpha=0.25 (hard example focus)')
print('  Contrastive  : margin=0.5 (pair-wise real/spoof separation)')


CombinedPADLoss ready:
  ArcFace      : m=0.50, s=30 (angular margin)
  Focal Loss   : gamma=2, alpha=0.25 (hard example focus)
  Contrastive  : margin=0.5 (pair-wise real/spoof separation)


In [14]:
# ── Cell 9: Training Helpers ──────────────────────────────────────
def run_epoch(model, criterion, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    with torch.set_grad_enabled(train):
        for rgb, patches, hf, fft, lbp, labels in loader:
            rgb     = rgb.to(DEVICE, non_blocking=True)
            patches = patches.to(DEVICE, non_blocking=True)
            hf      = hf.to(DEVICE, non_blocking=True)
            fft     = fft.to(DEVICE, non_blocking=True)
            lbp     = lbp.to(DEVICE, non_blocking=True)
            labels  = labels.to(DEVICE, non_blocking=True)
 
            logit, emb = model(rgb, patches, hf, fft, lbp)
            loss = criterion(logit, labels, embeddings=emb)
 
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
 
            total_loss += loss.item() * len(labels)
            all_preds.extend(torch.sigmoid(logit).detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / len(loader.dataset)
    auc  = roc_auc_score(all_labels, all_preds)
    acc  = ((np.array(all_preds) > 0.5) == np.array(all_labels)).mean()
    return avg_loss, auc, acc, np.array(all_preds), np.array(all_labels)
 
 
def find_optimal_threshold(preds, labels):
    """
    ─────────────────────────────────────────────────────────────────
    OPTIMAL THRESHOLD SELECTION:
 
    Why not use 0.5?
      Fixed threshold 0.5 works only if model is perfectly calibrated.
      In practice, the optimal decision boundary shifts based on:
        - Class imbalance (more attacks than real → shift threshold)
        - Loss function scaling (ArcFace changes score distribution)
        - Dataset difficulty (hard attacks → lower scores for real)
 
    How to find it?
      From the ROC curve, find the threshold that maximizes:
        Youden's J statistic = TPR - FPR = Sensitivity + Specificity - 1
      This is the point on the ROC curve furthest from the diagonal.
      Equivalently: maximizes (TPR - FPR), the "lift" over random.
 
      Alternative: minimize HTER = (FAR + FRR) / 2
      We use HTER minimization since it's the PAD evaluation metric.
 
    Effect:
      - Reduces FP (real faces wrongly rejected)
      - Reduces FN (spoof attacks that succeed)
      - ASR ↓ because threshold tightened for spoof direction
    ─────────────────────────────────────────────────────────────────
    """
    fpr, tpr, thresholds = roc_curve(labels, preds)
    fnr = 1 - tpr
 
    # Method 1: Minimize HTER = (FAR + FRR) / 2
    hter_arr = (fpr + fnr) / 2
    hter_idx = np.argmin(hter_arr)
    best_thr_hter = float(thresholds[hter_idx])
 
    # Method 2: Maximize Youden's J (TPR - FPR)
    j_arr    = tpr - fpr
    j_idx    = np.argmax(j_arr)
    best_thr_j = float(thresholds[j_idx])
 
    # Method 3: EER threshold (FAR = FRR)
    eer_idx = np.argmin(np.abs(fpr - fnr))
    best_thr_eer = float(thresholds[eer_idx])
 
    print(f'\n  Threshold candidates:')
    print(f'    Min HTER   threshold: {best_thr_hter:.4f}  (HTER={hter_arr[hter_idx]:.4f})')
    print(f'    Max Youden threshold: {best_thr_j:.4f}    (J={j_arr[j_idx]:.4f})')
    print(f'    EER        threshold: {best_thr_eer:.4f}  (EER={((fpr[eer_idx]+fnr[eer_idx])/2):.4f})')
 
    return best_thr_hter, best_thr_j, best_thr_eer, fpr, tpr, thresholds
 
 
def evaluate_test(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
 
    with torch.no_grad():
        for rgb, patches, hf, fft, lbp, labels in loader:
            rgb = rgb.to(DEVICE); patches = patches.to(DEVICE)
            hf  = hf.to(DEVICE);  fft = fft.to(DEVICE); lbp = lbp.to(DEVICE)
            logit, _ = model(rgb, patches, hf, fft, lbp)
            all_preds.extend(torch.sigmoid(logit).cpu().numpy())
            all_labels.extend(labels.numpy())
 
    preds_arr  = np.array(all_preds)
    labels_arr = np.array(all_labels).astype(int)
    preds_bin  = (preds_arr > threshold).astype(int)
 
    auc  = roc_auc_score(labels_arr, preds_arr)
    acc  = (preds_bin == labels_arr).mean()
    cm   = confusion_matrix(labels_arr, preds_bin)
    tn, fp, fn, tp = cm.ravel()
 
    far  = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr  = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    hter = (far + frr) / 2
    asr  = fp / (fn + tp) if (fn + tp) > 0 else 0.0
 
    fpr_a, tpr_a, _ = roc_curve(labels_arr, preds_arr)
    fnr_a = 1 - tpr_a
    eer   = float((fpr_a + fnr_a)[np.argmin(np.abs(fpr_a - fnr_a))]) / 2
 
    return dict(acc=acc, auc=auc, far=far, frr=frr, hter=hter, asr=asr,
                eer=eer, tn=tn, fp=fp, fn=fn, tp=tp,
                fpr=fpr_a, tpr=tpr_a, preds=preds_arr, labels=labels_arr,
                threshold=threshold)
 
print('Helpers ready.')


Helpers ready.


In [15]:
print(f"Is CUDA (GPU) available? {torch.cuda.is_available()}")
print(f"Current Device: {DEVICE}")

if DEVICE.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")

Is CUDA (GPU) available? True
Current Device: cuda
GPU Name: NVIDIA RTX A5000
Memory Allocated: 22.19 MB


In [16]:
import os

test_path = './test_write.txt'
try:
    with open(test_path, 'w') as f:
        f.write('Storage check')
    os.remove(test_path)
    print("✅ Success! You have permission to save models in this folder.")
except Exception as e:
    print(f"❌ Storage error: {e}")

✅ Success! You have permission to save models in this folder.


In [17]:
# -- Cell 10: Training -- 100 epochs per phase, NO early stopping --
history = []

# -- Phase 1: Heads only (RGB backbone frozen) --
print('='*62)
print('PHASE 1 -- Texture + LBP + Patch branches (RGB frozen)')
print('='*62)
model.freeze_backbone()
for name, p in model.named_parameters():
    if any(x in name for x in ['texture_branch','lbp_branch',
                                'patch_branch','rgb_proj','classifier']):
        p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable/1e6:.2f}M\n')

opt1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR1, weight_decay=1e-4
)

best_auc_p1 = 0.0

for ep in range(1, PHASE1_EP + 1):
    tr_loss, tr_auc, tr_acc, _, _ = run_epoch(
        model, criterion, train_loader, opt1, train=True)
    vl_loss, vl_auc, vl_acc, _, _ = run_epoch(
        model, criterion, val_loader, train=False)
    history.append(('p1', tr_loss, vl_loss, tr_auc, vl_auc))
    flag = ''
    if vl_auc > best_auc_p1:
        best_auc_p1 = vl_auc
        torch.save(model.state_dict(), './best_attn_pad.pt')
        flag = '  checkmark saved'
    print(f'Ep{ep:03d} | tr loss={tr_loss:.4f} auc={tr_auc:.4f} acc={tr_acc:.3f} | '
          f'val loss={vl_loss:.4f} auc={vl_auc:.4f} acc={vl_acc:.3f}' + flag)

print(f'\nPhase 1 done. Best val AUC: {best_auc_p1:.4f}')

# -- Phase 2: Fine-tune last 2 RGB blocks + all heads --
print('\n' + '='*62)
print('PHASE 2 -- Fine-tune last 2 EfficientNet blocks + all heads')
print('='*62)
model.load_state_dict(torch.load('./best_attn_pad.pt'))
model.unfreeze_last_blocks(n_blocks=2)
for name, p in model.named_parameters():
    if any(x in name for x in ['texture_branch','lbp_branch',
                                'patch_branch','rgb_proj','classifier']):
        p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable/1e6:.2f}M\n')

opt2  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR2, weight_decay=1e-4
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2, T_max=PHASE2_EP, eta_min=1e-6)

best_auc_p2 = 0.0
val_preds_best, val_labels_best = None, None  # initialise safely

for ep in range(1, PHASE2_EP + 1):
    tr_loss, tr_auc, tr_acc, _,  _ = run_epoch(
        model, criterion, train_loader, opt2, train=True)
    vl_loss, vl_auc, vl_acc, vp, vl = run_epoch(
        model, criterion, val_loader, train=False)
    sched.step()
    history.append(('p2', tr_loss, vl_loss, tr_auc, vl_auc))
    flag = ''
    if vl_auc > best_auc_p2:
        best_auc_p2 = vl_auc
        torch.save(model.state_dict(), './best_attn_pad.pt')
        val_preds_best  = vp.copy()
        val_labels_best = vl.copy()
        flag = '  checkmark saved'
    print(f'Ep{ep:03d} | tr loss={tr_loss:.4f} auc={tr_auc:.4f} acc={tr_acc:.3f} | '
          f'val loss={vl_loss:.4f} auc={vl_auc:.4f} acc={vl_acc:.3f}' + flag)

print(f'\nPhase 2 done. Best val AUC: {best_auc_p2:.4f}')

# Safety fallback: if Phase 2 had zero improvement (shouldn't happen)
if val_preds_best is None:
    print('WARNING: Running val inference to get threshold calibration data...')
    _, _, _, val_preds_best, val_labels_best = run_epoch(
        model, criterion, val_loader, train=False)


PHASE 1 -- Texture + LBP + Patch branches (RGB frozen)
Trainable: 1.74M

Ep001 | tr loss=3.1658 auc=0.9182 acc=0.865 | val loss=0.8333 auc=0.9921 acc=0.971  checkmark saved
Ep002 | tr loss=2.1863 auc=0.9286 acc=0.886 | val loss=0.8956 auc=0.9944 acc=0.972  checkmark saved
Ep003 | tr loss=1.8186 auc=0.9376 acc=0.894 | val loss=0.8952 auc=0.9910 acc=0.965
Ep004 | tr loss=1.6593 auc=0.9419 acc=0.901 | val loss=0.8066 auc=0.9964 acc=0.973  checkmark saved
Ep005 | tr loss=1.6274 auc=0.9455 acc=0.902 | val loss=1.1294 auc=0.9901 acc=0.954
Ep006 | tr loss=1.6141 auc=0.9417 acc=0.896 | val loss=0.9651 auc=0.9939 acc=0.965
Ep007 | tr loss=1.3377 auc=0.9514 acc=0.905 | val loss=0.8747 auc=0.9940 acc=0.970
Ep008 | tr loss=1.2723 auc=0.9517 acc=0.902 | val loss=0.6989 auc=0.9969 acc=0.951  checkmark saved
Ep009 | tr loss=1.1872 auc=0.9613 acc=0.909 | val loss=2.8698 auc=0.9895 acc=0.910
Ep010 | tr loss=1.1676 auc=0.9617 acc=0.907 | val loss=0.8893 auc=0.9946 acc=0.962
Ep011 | tr loss=1.1756 auc=0.

In [18]:
# ── Cell 11: Optimal Threshold + Final Evaluation ─────────────────
print('='*62)
print('OPTIMAL THRESHOLD SELECTION (from validation set)')
print('='*62)
 
model.load_state_dict(torch.load('./best_attn_pad.pt'))
 
thr_hter, thr_j, thr_eer, val_fpr, val_tpr, val_thrs = \
    find_optimal_threshold(val_preds_best, val_labels_best)
 
print(f'\n  Using HTER-optimal threshold: {thr_hter:.4f}')
 
print('\n' + '='*62)
print('FINAL TEST EVALUATION')
print('='*62)
 
# Evaluate with fixed 0.5 AND optimal threshold for comparison
res_05  = evaluate_test(model, test_loader, threshold=0.5)
res_opt = evaluate_test(model, test_loader, threshold=thr_hter)
 
print(f'\n  ┌───────────────┬──────────────┬──────────────────┐')
print(f'  │ Metric        │ threshold=0.5│ threshold={thr_hter:.3f} │')
print(f'  ├───────────────┼──────────────┼──────────────────┤')
for key, label in [('acc','Accuracy'), ('auc','AUC'), ('eer','EER'),
                   ('far','FAR'), ('frr','FRR'),
                   ('hter','HTER'), ('asr','ASR')]:
    v1 = res_05[key];  v2 = res_opt[key]
    mult = 100 if key in ('acc','eer','asr') else 1
    unit = '%' if key in ('acc','eer','asr') else ''
    print(f'  │ {label:<13} │ {v1*mult:>10.2f}{unit} │ {v2*mult:>14.2f}{unit}   │')
print(f'  └───────────────┴──────────────┴──────────────────┘')
 
# Use optimal for remaining outputs
results = res_opt
 

OPTIMAL THRESHOLD SELECTION (from validation set)

  Threshold candidates:
    Min HTER   threshold: 0.5492  (HTER=0.0165)
    Max Youden threshold: 0.7428    (J=0.9670)
    EER        threshold: 0.7428  (EER=0.0165)

  Using HTER-optimal threshold: 0.5492

FINAL TEST EVALUATION

  ┌───────────────┬──────────────┬──────────────────┐
  │ Metric        │ threshold=0.5│ threshold=0.549 │
  ├───────────────┼──────────────┼──────────────────┤
  │ Accuracy      │      96.67% │          96.61%   │
  │ AUC           │       0.99 │           0.99   │
  │ EER           │       3.00% │           3.00%   │
  │ FAR           │       0.01 │           0.01   │
  │ FRR           │       0.05 │           0.06   │
  │ HTER          │       0.03 │           0.03   │
  │ ASR           │       1.22% │           1.00%   │
  └───────────────┴──────────────┴──────────────────┘


In [19]:
# ── Cell 12: Comprehensive Visualization ─────────────────────────
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('AttentionPAD — SE Blocks + Patches + Optimal Threshold\n'
             'EfficientNet-B0 + TextureCNN(SE) + LBP + PatchBranch',
             fontsize=14, fontweight='bold')
 
epochs_all = range(1, len(history) + 1)
p1_end     = sum(1 for h in history if h[0] == 'p1')
tr_losses  = [h[1] for h in history]
vl_losses  = [h[2] for h in history]
tr_aucs    = [h[3] for h in history]
vl_aucs    = [h[4] for h in history]
 
# 1. Loss
ax = fig.add_subplot(gs[0, 0])
ax.plot(epochs_all, tr_losses, color='steelblue', lw=2, label='Train')
ax.plot(epochs_all, vl_losses, color='coral',     lw=2, label='Val', linestyle='--')
ax.axvline(p1_end + 0.5, color='gray', linestyle=':', alpha=0.6, label='P1→P2')
ax.set_title('Training Loss', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlabel('Epoch')
 
# 2. AUC
ax = fig.add_subplot(gs[0, 1])
ax.plot(epochs_all, tr_aucs, color='steelblue', lw=2, label='Train')
ax.plot(epochs_all, vl_aucs, color='coral',     lw=2, label='Val',   linestyle='--')
ax.axvline(p1_end + 0.5, color='gray', linestyle=':', alpha=0.6)
ax.set_ylim([0.5, 1.02])
ax.set_title('AUC', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlabel('Epoch')
 
# 3. ROC with both thresholds marked
ax = fig.add_subplot(gs[0, 2])
ax.plot(results['fpr'], results['tpr'], color='steelblue', lw=2.5,
        label=f'AUC={results["auc"]:.4f}')
ax.fill_between(results['fpr'], results['tpr'], alpha=0.08, color='steelblue')
ax.plot([0,1],[0,1],'k--', lw=1)
# Mark thresholds on ROC curve
for thr, col, lbl in [(thr_hter,'red','HTER opt'),
                       (thr_eer, 'green', 'EER')]:
    preds_b = (results['preds'] > thr).astype(int)
    tn_,fp_,fn_,tp_ = confusion_matrix(results['labels'], preds_b).ravel()
    fpr_ = fp_/(fp_+tn_+1e-7); tpr_ = tp_/(tp_+fn_+1e-7)
    ax.scatter(fpr_, tpr_, color=col, s=80, zorder=5, label=f'τ={thr:.3f} ({lbl})')
ax.set_xlabel('FAR'); ax.set_ylabel('TAR')
ax.set_title('ROC — Threshold Comparison', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
 
# 4. Confusion matrix (optimal threshold)
ax = fig.add_subplot(gs[0, 3])
cm_data = np.array([[results['tn'], results['fp']],
                    [results['fn'], results['tp']]])
im = ax.imshow(cm_data, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_data[i,j]), ha='center', va='center',
                fontsize=12, fontweight='bold',
                color='white' if cm_data[i,j] > cm_data.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred Real','Pred Attack'], fontsize=8)
ax.set_yticklabels(['Actual Real','Actual Attack'], fontsize=8)
ax.set_title(f'Confusion Matrix\n(τ={thr_hter:.3f})', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
 
# 5. Threshold sweep — HTER vs threshold
ax = fig.add_subplot(gs[1, :2])
fpr_v, tpr_v, thrs_v = roc_curve(val_labels_best, val_preds_best)
fnr_v   = 1 - tpr_v
hter_v  = (fpr_v + fnr_v) / 2
ax.plot(thrs_v, hter_v[:-1] if len(thrs_v) == len(hter_v)-1 else hter_v,
        color='steelblue', lw=2, label='HTER')
ax.plot(thrs_v, fpr_v[:-1]  if len(thrs_v) == len(fpr_v)-1  else fpr_v,
        color='coral',     lw=2, linestyle='--', label='FAR')
ax.plot(thrs_v, fnr_v[:-1]  if len(thrs_v) == len(fnr_v)-1  else fnr_v,
        color='green',     lw=2, linestyle='--', label='FRR')
ax.axvline(thr_hter, color='red', linestyle=':', lw=2,
           label=f'Optimal τ={thr_hter:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Error Rate')
ax.set_title('Threshold Sweep — Val Set\n(select τ that minimizes HTER)',
             fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
 
# 6. Metric comparison: before vs after optimal threshold
ax = fig.add_subplot(gs[1, 2:])
metric_labels = ['Accuracy%','AUC×100','EER%','FAR%','FRR%','HTER%','ASR%']
v_05  = [res_05['acc']*100,  res_05['auc']*100,  res_05['eer']*100,
         res_05['far']*100,  res_05['frr']*100,  res_05['hter']*100,
         res_05['asr']*100]
v_opt = [res_opt['acc']*100, res_opt['auc']*100, res_opt['eer']*100,
         res_opt['far']*100, res_opt['frr']*100, res_opt['hter']*100,
         res_opt['asr']*100]
x     = np.arange(len(metric_labels))
w     = 0.38
ax.bar(x - w/2, v_05,  w, label='τ=0.5 (fixed)',  color='coral',     alpha=0.8)
ax.bar(x + w/2, v_opt, w, label=f'τ={thr_hter:.3f} (optimal)', color='steelblue', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(metric_labels, rotation=20, ha='right', fontsize=9)
ax.set_title('Fixed vs Optimal Threshold', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
 
# 7. Architecture summary
ax = fig.add_subplot(gs[2, :])
ax.axis('off')
arch = (
    "ARCHITECTURE SUMMARY\n"
    "─────────────────────────────────────────────────────────────────────\n"
    "Branch 1: RGB (3×224×224)    → EfficientNet-B0              → 256-d\n"
    "Branch 2: HF+FFT (4×224×224) → TextureCNN + SE(32) + SE(64) + SE(128) → 128-d  ← ATTENTION\n"
    "Branch 3: LBP hist (256,)    → MLP                          →  64-d\n"
    "Branch 4: 4×patch(3×112×112) → PatchCNN → mean+max pool     →  64-d  ← PATCH LEARNING\n"
    "                               CONCAT → 512-d → L2-norm → Classifier → logit\n"
    "─────────────────────────────────────────────────────────────────────\n"
    "LOSS: ArcFace(m=0.5,s=30) + 0.5×Focal(γ=2) + 0.3×Contrastive(margin=0.5)\n"
    "THRESHOLD: Not fixed at 0.5 — selected from val set by minimizing HTER\n"
    "COSINE: All embeddings L2-normalized → similarity ∈ [-1,+1], not L2=64"
)
ax.text(0.01, 0.5, arch, transform=ax.transAxes, fontsize=9.5,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f0f4ff', alpha=0.9))
 
plt.savefig('./attention_pad_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ./attention_pad_results.png')
 

Saved → ./attention_pad_results.png


In [20]:
# ── Cell 13: Save Everything ──────────────────────────────────────
shutil.copy('./best_attn_pad.pt',
            f'{SAVE_DIR}/attention_pad_best.pt')
shutil.copy('./attention_pad_results.png',
            f'{SAVE_DIR}/attention_pad_results.png')
 
with open(f'{SAVE_DIR}/attention_pad_metrics.txt', 'w') as f:
    f.write('AttentionPAD — SE + Patches + Optimal Threshold\n')
    f.write('='*55 + '\n')
    f.write(f'Optimal threshold (val HTER): {thr_hter:.4f}\n\n')
    for label, res, thr in [
        ('Fixed threshold (0.5)',    res_05,  0.5),
        (f'Optimal threshold ({thr_hter:.3f})', res_opt, thr_hter)
    ]:
        f.write(f'{label}:\n')
        f.write(f'  Accuracy: {res["acc"]*100:.2f}%\n')
        f.write(f'  AUC     : {res["auc"]:.4f}\n')
        f.write(f'  EER     : {res["eer"]*100:.2f}%\n')
        f.write(f'  FAR     : {res["far"]:.4f}\n')
        f.write(f'  FRR     : {res["frr"]:.4f}\n')
        f.write(f'  HTER    : {res["hter"]:.4f}\n')
        f.write(f'  ASR     : {res["asr"]*100:.2f}%\n')
        f.write(f'  TN={res["tn"]}  FP={res["fp"]}  FN={res["fn"]}  TP={res["tp"]}\n\n')
 
print(f'Saved to {SAVE_DIR}')
print('  attention_pad_best.pt')
print('  attention_pad_results.png')
print('  attention_pad_metrics.txt')
print('\nDone!')

Saved to DL_ProjectP12_clean/outputs
  attention_pad_best.pt
  attention_pad_results.png
  attention_pad_metrics.txt

Done!
